In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import koreanize_matplotlib
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
# from my_ml_kit import *
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import GridSearchCV
# =========================
# Classification Models
# =========================

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    HistGradientBoostingClassifier,
    VotingClassifier,
    StackingClassifier,
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


# =========================
# Regression Models
# =========================

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
)

from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor,
    HistGradientBoostingRegressor,
    VotingRegressor,
    StackingRegressor,
)

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [ ]:
train_data = pd.read_csv("train.csv",na_values=['None', 'none', 'NULL', 'null'])
test_data = pd.read_csv("test.csv")
submission_data = pd.read_csv("sample_submission.csv")

In [ ]:
train_data.head()

In [ ]:
train_data.info()

In [ ]:
train_data.isnull().sum()

In [ ]:
time_cols = [
    '임신 시도 또는 마지막 임신 경과 연수',
    '난자 해동 경과일',
    '난자 혼합 경과일',
    '배아 이식 경과일',
    '배아 해동 경과일',
]

for col in time_cols:
    train_data[f'{col}_performed'] = (
        train_data[col].notnull()
    ).astype(int)


train_data[time_cols] = train_data[time_cols].fillna(0)

In [ ]:
train_data = train_data.fillna(0)

In [ ]:
train_data.isnull().sum()

In [ ]:
train_data.info()

In [ ]:
train_sub = train_data.copy()
X_sub = train_sub.drop(["임신 성공 여부","ID"],axis=1)
y_sub = train_sub["임신 성공 여부"]

In [ ]:
binary_cols = []

for col in X_sub.columns:
    unique_values = set(X_sub[col].dropna().unique())

    if unique_values == {0, 1}:
        binary_cols.append(col)

X_sub[binary_cols] = X_sub[binary_cols].astype('category')

In [ ]:
X_sub.info()

In [ ]:
to_categorical_columns  = ["불임 원인 - 여성 요인","착상 전 유전 검사 사용 여부","PGD 시술 여부","PGS 시술 여부"]
X_sub[to_categorical_columns] = X_sub[to_categorical_columns].astype('category')
str_columns = X_sub.select_dtypes("str").columns
X_sub[str_columns] = X_sub[str_columns].astype('category')
X_sub.info()

In [ ]:
categorical_columns = X_sub.select_dtypes("category").columns
numeric_columns = X_sub.select_dtypes("float64").columns

for cat_column in categorical_columns:
    print(f"{cat_column}의 고윳값 : \n {X_sub[cat_column].unique()} \n =====================================================")

In [ ]:
y_sub.value_counts()

In [ ]:
print("Numeric Columns 단변량 분석\n")
for col in numeric_columns:
    plt.figure(figsize=(12, 4))

    # Histogram
    plt.subplot(1, 2, 1)
    sns.histplot(
        train_sub[col],
        kde=True
    )
    plt.title(f'{col} Histogram')

    # Boxplot
    plt.subplot(1, 2, 2)
    sns.boxplot(
        x=train_sub[col]
    )
    plt.title(f'{col} Boxplot')

    plt.tight_layout()
    plt.show()

print("="*50)
print("categorical columns 단변량 분석\n")
for col in categorical_columns:
    plt.figure(figsize=(8, 4))

    sns.countplot(
        x=train_sub[col],
        order=train_sub[col].value_counts().index
    )

    plt.title(f'{col} Countplot')
    plt.xticks(rotation=45)

    plt.tight_layout()
    plt.show()

In [ ]:
categorical_features = X_sub.select_dtypes("category").columns
numeric_features = X_sub.select_dtypes("float64").columns

numeric_transformer = Pipeline([
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

In [ ]:
classification_basic_models = {
    "logistic_regression": LogisticRegression(max_iter=1000),
    "knn_classifier": KNeighborsClassifier(),
    "decision_tree_classifier": DecisionTreeClassifier(random_state=42),
    "gaussian_nb": GaussianNB(),
}

In [ ]:
data_split = make_data_split(
    X_sub,
    y_sub,
    test_size=0.2,
    stratify=y_sub,
    random_state=42,
)

In [ ]:
results, trained_pipes, predictions = run_model_candidates(
    models=classification_basic_models,
    preprocessor=preprocessor,
    **data_split,
    task_type="classification",
)

display(compare_results(results))

In [ ]:
count_cols = [
    '총 시술 횟수',
    'IVF 시술 횟수',
    'DI 시술 횟수',
    '총 임신 횟수',
    'IVF 임신 횟수',
    'DI 임신 횟수',
    '총 출산 횟수',
    'IVF 출산 횟수',
    'DI 출산 횟수',
]

count_map = {
    '0회': 0,
    '1회': 1,
    '2회': 2,
    '3회': 3,
    '4회': 4,
    '5회': 5,
    '6회 이상': 6,
}

# ordinal 변환
for col in count_cols:
    X_sub[col] = X_sub[col].map(count_map).astype(float)

# -----------------------------
# 성공률 feature
# -----------------------------

X_sub['총_임신_성공률'] = np.where(
    X_sub['총 시술 횟수'] == 0,
    0,
    X_sub['총 임신 횟수'] / X_sub['총 시술 횟수']
)

X_sub['IVF_임신_성공률'] = np.where(
    X_sub['IVF 시술 횟수'] == 0,
    0,
    X_sub['IVF 임신 횟수'] / X_sub['IVF 시술 횟수']
)

X_sub['DI_임신_성공률'] = np.where(
    X_sub['DI 시술 횟수'] == 0,
    0,
    X_sub['DI 임신 횟수'] / X_sub['DI 시술 횟수']
)

# -----------------------------
# 출산 전환률
# -----------------------------

X_sub['총_출산_전환률'] = np.where(
    X_sub['총 임신 횟수'] == 0,
    0,
    X_sub['총 출산 횟수'] / X_sub['총 임신 횟수']
)

X_sub['IVF_출산_전환률'] = np.where(
    X_sub['IVF 임신 횟수'] == 0,
    0,
    X_sub['IVF 출산 횟수'] / X_sub['IVF 임신 횟수']
)

X_sub['DI_출산_전환률'] = np.where(
    X_sub['DI 임신 횟수'] == 0,
    0,
    X_sub['DI 출산 횟수'] / X_sub['DI 임신 횟수']
)

# -----------------------------
# 실패 횟수
# -----------------------------

X_sub['총_실패_횟수'] = (
    X_sub['총 시술 횟수'] -
    X_sub['총 임신 횟수']
)

X_sub['IVF_실패_횟수'] = (
    X_sub['IVF 시술 횟수'] -
    X_sub['IVF 임신 횟수']
)

X_sub['DI_실패_횟수'] = (
    X_sub['DI 시술 횟수'] -
    X_sub['DI 임신 횟수']
)

# -----------------------------
# IVF 비중
# -----------------------------

X_sub['IVF_비중'] = np.where(
    X_sub['총 시술 횟수'] == 0,
    0,
    X_sub['IVF 시술 횟수'] / X_sub['총 시술 횟수']
)

X_sub['DI_비중'] = np.where(
    X_sub['총 시술 횟수'] == 0,
    0,
    X_sub['DI 시술 횟수'] / X_sub['총 시술 횟수']
)
X_sub['배아_이식률'] = np.where(
    X_sub['총 생성 배아 수'] == 0,
    0,
    X_sub['이식된 배아 수'] / X_sub['총 생성 배아 수']
)
X_sub['배아_저장률'] = np.where(
    X_sub['총 생성 배아 수'] == 0,
    0,
    X_sub['저장된 배아 수'] / X_sub['총 생성 배아 수']
)
X_sub['난자_배아_효율'] = np.where(
    X_sub['수집된 신선 난자 수'] == 0,
    0,
    X_sub['총 생성 배아 수'] / X_sub['수집된 신선 난자 수']
)
X_sub['총_배아_활용량'] = (
    X_sub['이식된 배아 수'] +
    X_sub['저장된 배아 수']
)
X_sub['배아이식_수행여부'] = (
    X_sub['이식된 배아 수'] > 0
).astype(int)
X_sub['고령여부'] = X_sub['시술 당시 나이'].isin([
    '만38-39세',
    '만40-42세',
    '만43-44세',
    '만45-50세'
]).astype(int)
X_sub['고령_이식배아_interaction'] = (
    X_sub['고령여부'] *
    X_sub['이식된 배아 수']
)
X_sub['고령_생성배아_interaction'] = (
    X_sub['고령여부'] *
    X_sub['총 생성 배아 수']
)
X_sub['고령_저장배아_interaction'] = (
    X_sub['고령여부'] *
    X_sub['저장된 배아 수']
)
X_sub['고령_난자수_interaction'] = (
    X_sub['고령여부'] *
    X_sub['수집된 신선 난자 수']
)
X_sub['고령_배아이식률_interaction'] = (
    X_sub['고령여부'] *
    X_sub['배아_이식률']
)

In [ ]:
X_sub.info()

In [ ]:
numeric_features = X_sub.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_sub.select_dtypes(include="category").columns.tolist()

numeric_transformer = Pipeline([
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
])

In [ ]:
classification_basic_models = {
    "logistic_regression": LogisticRegression(max_iter=1000,class_weight='balanced'),
    # "knn_classifier": KNeighborsClassifier(),
    # "decision_tree_classifier": DecisionTreeClassifier(random_state=42),
    # "gaussian_nb": GaussianNB(),
}
data_split = make_data_split(
    X_sub,
    y_sub,
    test_size=0.2,
    stratify=y_sub,
    random_state=42,
)
results, trained_pipes, predictions = run_model_candidates(
    models=classification_basic_models,
    preprocessor=preprocessor,
    **data_split,
    task_type="classification",
)

display(compare_results(results))

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

pipe = trained_pipes["logistic_regression"]

X_test = data_split["X_test"]
y_test = data_split["y_test"]

y_proba = pipe.predict_proba(X_test)[:, 1]

threshold_results = []

for threshold in np.arange(0.1, 0.91, 0.01):
    y_pred_threshold = (y_proba >= threshold).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "accuracy": accuracy_score(y_test, y_pred_threshold),
        "f1": f1_score(y_test, y_pred_threshold),
        "precision": precision_score(y_test, y_pred_threshold),
        "recall": recall_score(y_test, y_pred_threshold),
    })

df_threshold = pd.DataFrame(threshold_results)

df_threshold.sort_values("f1", ascending=False).head(10)

In [ ]:
best_threshold = df_threshold.sort_values("f1", ascending=False).iloc[0]["threshold"]
y_pred_best = (y_proba >= best_threshold).astype(int)

print("best_threshold:", best_threshold)
print("f1:", f1_score(y_test, y_pred_best))
print("precision:", precision_score(y_test, y_pred_best))
print("recall:", recall_score(y_test, y_pred_best))
print("accuracy:", accuracy_score(y_test, y_pred_best))
print(confusion_matrix(y_test, y_pred_best))
print(classification_report(y_test, y_pred_best))

In [ ]:
X_train = data_split["X_train"]
X_test = data_split["X_test"]
y_train = data_split["y_train"]
y_test = data_split["y_test"]

X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)

cat_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="F1",
    random_seed=42,
    verbose=100,
    class_weights=[1, 190123 / 66228]
)

cat_model.fit(
    X_train_trans,
    y_train,
    eval_set=(X_test_trans, y_test),
    early_stopping_rounds=50
)

y_pred = cat_model.predict(X_test_trans)
y_proba = cat_model.predict_proba(X_test_trans)[:, 1]

print("accuracy:", accuracy_score(y_test, y_pred))
print("f1:", f1_score(y_test, y_pred))
print("precision:", precision_score(y_test, y_pred))
print("recall:", recall_score(y_test, y_pred))
print("roc_auc:", roc_auc_score(y_test, y_proba))

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
feature_names = preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": cat_model.feature_importances_
})

importance_df.sort_values(
    "importance",
    ascending=False
).head(30)

In [ ]:
data_split = make_data_split(
    X_sub,
    y_sub,
    test_size=0.2,
    stratify=y_sub,
    random_state=42,
)

X_train = data_split["X_train"]
X_test = data_split["X_test"]
y_train = data_split["y_train"]
y_test = data_split["y_test"]

X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)

cat_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="F1",
    random_seed=42,
    verbose=100,
    class_weights=[1, 190123 / 66228]
)

cat_model.fit(
    X_train_trans,
    y_train,
    eval_set=(X_test_trans, y_test),
    early_stopping_rounds=50
)

y_pred = cat_model.predict(X_test_trans)
y_proba = cat_model.predict_proba(X_test_trans)[:, 1]

print("accuracy:", accuracy_score(y_test, y_pred))
print("f1:", f1_score(y_test, y_pred))
print("precision:", precision_score(y_test, y_pred))
print("recall:", recall_score(y_test, y_pred))
print("roc_auc:", roc_auc_score(y_test, y_proba))

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
X_sub.columns[
    X_sub.columns.str.contains("interaction")
]

In [ ]:
numeric_features = X_sub.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_sub.select_dtypes(include="category").columns.tolist()

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
])

data_split = make_data_split(
    X_sub,
    y_sub,
    test_size=0.2,
    stratify=y_sub,
    random_state=42,
)

X_train = data_split["X_train"]
X_test = data_split["X_test"]
y_train = data_split["y_train"]
y_test = data_split["y_test"]

X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)

cat_model.fit(
    X_train_trans,
    y_train,
    eval_set=(X_test_trans, y_test),
    early_stopping_rounds=50
)

y_pred = cat_model.predict(X_test_trans)
y_proba = cat_model.predict_proba(X_test_trans)[:, 1]

print("f1:", f1_score(y_test, y_pred))
print("precision:", precision_score(y_test, y_pred))
print("recall:", recall_score(y_test, y_pred))
print("roc_auc:", roc_auc_score(y_test, y_proba))

In [ ]:
feature_names = preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": cat_model.feature_importances_
})

importance_df.sort_values("importance", ascending=False).head(30)

In [ ]:
X_sub['배아_이식_집중도'] = np.where(
    (X_sub['이식된 배아 수'] + X_sub['저장된 배아 수']) == 0,
    0,
    X_sub['이식된 배아 수'] /
    (
        X_sub['이식된 배아 수'] +
        X_sub['저장된 배아 수']
    )
)
X_sub['미활용_배아수'] = (
    X_sub['총 생성 배아 수']
    - X_sub['이식된 배아 수']
    - X_sub['저장된 배아 수']
)
X_sub['배아_손실률'] = np.where(
    X_sub['총 생성 배아 수'] == 0,
    0,
    1 - (
        (
            X_sub['이식된 배아 수']
            + X_sub['저장된 배아 수']
        )
        / X_sub['총 생성 배아 수']
    )
)

In [ ]:
numeric_features = X_sub.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_sub.select_dtypes(include="category").columns.tolist()

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
])

data_split = make_data_split(
    X_sub,
    y_sub,
    test_size=0.2,
    stratify=y_sub,
    random_state=42,
)

X_train = data_split["X_train"]
X_test = data_split["X_test"]
y_train = data_split["y_train"]
y_test = data_split["y_test"]

X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)

cat_model.fit(
    X_train_trans,
    y_train,
    eval_set=(X_test_trans, y_test),
    early_stopping_rounds=50
)

y_pred = cat_model.predict(X_test_trans)
y_proba = cat_model.predict_proba(X_test_trans)[:, 1]

print("f1:", f1_score(y_test, y_pred))
print("precision:", precision_score(y_test, y_pred))
print("recall:", recall_score(y_test, y_pred))
print("roc_auc:", roc_auc_score(y_test, y_proba))

In [ ]:
feature_names = preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": cat_model.feature_importances_
})

importance_df.sort_values("importance", ascending=False).head(30)

In [ ]:
X_sub['저장대비_이식비율'] = (
    X_sub['저장된 배아 수']
    / (X_sub['이식된 배아 수'] + 1)
)
X_sub['배아_활용효율'] = np.where(
    X_sub['총 생성 배아 수'] == 0,
    0,
    (
        X_sub['이식된 배아 수']
        + X_sub['저장된 배아 수']
    ) / X_sub['총 생성 배아 수']
)

In [ ]:
numeric_features = X_sub.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_sub.select_dtypes(include="category").columns.tolist()

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
])

data_split = make_data_split(
    X_sub,
    y_sub,
    test_size=0.2,
    stratify=y_sub,
    random_state=42,
)

X_train = data_split["X_train"]
X_test = data_split["X_test"]
y_train = data_split["y_train"]
y_test = data_split["y_test"]

X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)

cat_model.fit(
    X_train_trans,
    y_train,
    eval_set=(X_test_trans, y_test),
    early_stopping_rounds=50
)

y_pred = cat_model.predict(X_test_trans)
y_proba = cat_model.predict_proba(X_test_trans)[:, 1]

print("f1:", f1_score(y_test, y_pred))
print("precision:", precision_score(y_test, y_pred))
print("recall:", recall_score(y_test, y_pred))
print("roc_auc:", roc_auc_score(y_test, y_proba))

In [ ]:
feature_names = preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": cat_model.feature_importances_
})

importance_df.sort_values("importance", ascending=False).head(30)

In [ ]:
X_raw = train_data.drop(["임신 성공 여부", "ID"], axis=1).copy()
y_raw = train_data["임신 성공 여부"].copy()

# 문자형 category 변환
str_columns = X_raw.select_dtypes(include="object").columns
X_raw[str_columns] = X_raw[str_columns].astype("category")

# 0/1 컬럼 category 변환
binary_cols = []

for col in X_raw.columns:
    unique_values = set(X_raw[col].dropna().unique())

    if unique_values == {0, 1}:
        binary_cols.append(col)

X_raw[binary_cols] = X_raw[binary_cols].astype("category")

# feature 분리
numeric_features_raw = X_raw.select_dtypes(include=np.number).columns.tolist()
categorical_features_raw = X_raw.select_dtypes(include="category").columns.tolist()

numeric_transformer = Pipeline([
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
categorical_features_raw = X_raw.select_dtypes(
    include="category"
).columns.tolist()

X_raw[categorical_features_raw] = (
    X_raw[categorical_features_raw]
    .astype(str)
)
preprocessor_raw = ColumnTransformer([
    ("num", numeric_transformer, numeric_features_raw),
    ("cat", categorical_transformer, categorical_features_raw),
])
# =============================
# 같은 조건으로 split
# =============================

data_split_raw = make_data_split(
    X_raw,
    y_raw,
    test_size=0.2,
    stratify=y_raw,
    random_state=42,
)

X_train_raw = data_split_raw["X_train"]
X_test_raw = data_split_raw["X_test"]
y_train_raw = data_split_raw["y_train"]
y_test_raw = data_split_raw["y_test"]

X_train_raw_trans = preprocessor_raw.fit_transform(X_train_raw)
X_test_raw_trans = preprocessor_raw.transform(X_test_raw)
# =============================
# Raw CatBoost 학습
# =============================

cat_model_raw = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="F1",
    random_seed=42,
    verbose=100,
    class_weights=[1, 190123 / 66228]
)

cat_model_raw.fit(
    X_train_raw_trans,
    y_train_raw,
    eval_set=(X_test_raw_trans, y_test_raw),
    early_stopping_rounds=50
)

y_pred_raw = cat_model_raw.predict(X_test_raw_trans)
y_proba_raw = cat_model_raw.predict_proba(X_test_raw_trans)[:, 1]

print("RAW CatBoost")
print("accuracy:", accuracy_score(y_test_raw, y_pred_raw))
print("f1:", f1_score(y_test_raw, y_pred_raw))
print("precision:", precision_score(y_test_raw, y_pred_raw))
print("recall:", recall_score(y_test_raw, y_pred_raw))
print("roc_auc:", roc_auc_score(y_test_raw, y_proba_raw))

print(confusion_matrix(y_test_raw, y_pred_raw))
print(classification_report(y_test_raw, y_pred_raw))
feature_names_raw = preprocessor_raw.get_feature_names_out()

importance_raw_df = pd.DataFrame({
    "feature": feature_names_raw,
    "importance": cat_model_raw.feature_importances_
})

importance_raw_df.sort_values("importance", ascending=False).head(30)

In [ ]:
drop_interaction_cols = [
    '고령_이식배아_interaction',
    '고령_생성배아_interaction',
    '고령_저장배아_interaction',
    '고령_배아이식률_interaction',
]

X_sub = X_sub.drop(columns=drop_interaction_cols, errors='ignore')

In [ ]:
numeric_features = X_sub.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_sub.select_dtypes(include="category").columns.tolist()

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
])

In [ ]:
data_split = make_data_split(
    X_sub,
    y_sub,
    test_size=0.2,
    stratify=y_sub,
    random_state=42,
)

In [ ]:
X_train = data_split["X_train"]
X_test = data_split["X_test"]
y_train = data_split["y_train"]
y_test = data_split["y_test"]

X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)

cat_model.fit(
    X_train_trans,
    y_train,
    eval_set=(X_test_trans, y_test),
    early_stopping_rounds=50
)

y_pred = cat_model.predict(X_test_trans)
y_proba = cat_model.predict_proba(X_test_trans)[:, 1]

print("f1:", f1_score(y_test, y_pred))
print("precision:", precision_score(y_test, y_pred))
print("recall:", recall_score(y_test, y_pred))
print("roc_auc:", roc_auc_score(y_test, y_proba))

In [ ]:
feature_names = preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": cat_model.feature_importances_
})

importance_df.sort_values("importance", ascending=False).head(30)

In [ ]:
import shap

explainer = shap.TreeExplainer(cat_model)

shap_values = explainer.shap_values(X_test_trans)

feature_names = preprocessor.get_feature_names_out()

shap.summary_plot(
    shap_values,
    X_test_trans,
    feature_names=feature_names
)

In [ ]:
shap.summary_plot(
    shap_values,
    X_test_trans,
    feature_names=feature_names,
    plot_type="bar"
)

In [ ]:
X_sub['배아_활용효율'] = np.where(
    X_sub['총 생성 배아 수'] == 0,
    0,
    (
        X_sub['이식된 배아 수']
        + X_sub['저장된 배아 수']
    ) / X_sub['총 생성 배아 수']
)
X_sub['미활용_배아수'] = (
    X_sub['총 생성 배아 수']
    - X_sub['이식된 배아 수']
    - X_sub['저장된 배아 수']
)
X_sub['배아_손실률'] = np.where(
    X_sub['총 생성 배아 수'] == 0,
    0,
    1 - (
        (
            X_sub['이식된 배아 수']
            + X_sub['저장된 배아 수']
        ) / X_sub['총 생성 배아 수']
    )
)
X_sub['저장대비_이식비율'] = (
    X_sub['저장된 배아 수']
    / (X_sub['이식된 배아 수'] + 1)
)
X_sub['배아_생성효율'] = np.where(
    X_sub['수집된 신선 난자 수'] == 0,
    0,
    X_sub['총 생성 배아 수']
    / X_sub['수집된 신선 난자 수']
)

In [ ]:
numeric_features = X_sub.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_sub.select_dtypes(include="category").columns.tolist()

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
])

data_split = make_data_split(
    X_sub,
    y_sub,
    test_size=0.2,
    stratify=y_sub,
    random_state=42,
)

X_train = data_split["X_train"]
X_test = data_split["X_test"]
y_train = data_split["y_train"]
y_test = data_split["y_test"]

X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)

cat_model.fit(
    X_train_trans,
    y_train,
    eval_set=(X_test_trans, y_test),
    early_stopping_rounds=50
)

y_pred = cat_model.predict(X_test_trans)
y_proba = cat_model.predict_proba(X_test_trans)[:, 1]

print("f1:", f1_score(y_test, y_pred))
print("precision:", precision_score(y_test, y_pred))
print("recall:", recall_score(y_test, y_pred))
print("roc_auc:", roc_auc_score(y_test, y_proba))

In [ ]:
feature_names = preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": cat_model.feature_importances_
})

importance_df.sort_values("importance", ascending=False).head(30)

In [ ]:
process_cols = [
    '난자 해동 경과일_performed',
    '난자 혼합 경과일_performed',
    '배아 이식 경과일_performed',
    '배아 해동 경과일_performed'
]

X_sub[process_cols] = X_sub[process_cols].astype(int)

X_sub['시술_진행단계_수'] = (
    X_sub[process_cols]
    .sum(axis=1)
)
time_cols = [
    '난자 해동 경과일',
    '난자 혼합 경과일',
    '배아 이식 경과일',
    '배아 해동 경과일'
]

X_sub['총_시술_경과일'] = X_sub[time_cols].sum(axis=1)

X_sub['고위험_연령군'] = X_sub['시술 당시 나이'].isin([
    '만40-42세',
    '만43-44세',
    '만45-50세'
]).astype(int)

In [ ]:
numeric_features = X_sub.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_sub.select_dtypes(include="category").columns.tolist()

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
])

data_split = make_data_split(
    X_sub,
    y_sub,
    test_size=0.2,
    stratify=y_sub,
    random_state=42,
)

X_train = data_split["X_train"]
X_test = data_split["X_test"]
y_train = data_split["y_train"]
y_test = data_split["y_test"]

X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)

cat_model.fit(
    X_train_trans,
    y_train,
    eval_set=(X_test_trans, y_test),
    early_stopping_rounds=50
)

y_pred = cat_model.predict(X_test_trans)
y_proba = cat_model.predict_proba(X_test_trans)[:, 1]

print("f1:", f1_score(y_test, y_pred))
print("precision:", precision_score(y_test, y_pred))
print("recall:", recall_score(y_test, y_pred))
print("roc_auc:", roc_auc_score(y_test, y_proba))

In [ ]:
feature_names = preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": cat_model.feature_importances_
})

importance_df.sort_values("importance", ascending=False).head(30)

In [ ]:
important_features = [
    "num__배아_이식_집중도",
    "num__이식된 배아 수",
    "num__배아이식_수행여부",
    "num__고령_난자수_interaction",
    "num__배아 이식 경과일",
]

X_test_dense = X_test_trans

if hasattr(X_test_dense, "toarray"):
    X_test_dense = X_test_dense.toarray()

feature_names = preprocessor.get_feature_names_out()

for feature in important_features:
    shap.dependence_plot(
        feature,
        shap_values,
        X_test_dense,
        feature_names=feature_names
    )

In [ ]:
X_sub['다중_배아이식'] = (
    X_sub['이식된 배아 수'] >= 2
).astype(int)

X_sub['고도_시술진행군'] = (
    X_sub['시술_진행단계_수'] >= 3
).astype(int)

X_sub['적극_배아이식군'] = (
    X_sub['배아_이식_집중도'] >= 0.7
).astype(int)

In [ ]:
numeric_features = X_sub.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_sub.select_dtypes(include="category").columns.tolist()

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
])

data_split = make_data_split(
    X_sub,
    y_sub,
    test_size=0.2,
    stratify=y_sub,
    random_state=42,
)

X_train = data_split["X_train"]
X_test = data_split["X_test"]
y_train = data_split["y_train"]
y_test = data_split["y_test"]

X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)

cat_model.fit(
    X_train_trans,
    y_train,
    eval_set=(X_test_trans, y_test),
    early_stopping_rounds=50
)

y_pred = cat_model.predict(X_test_trans)
y_proba = cat_model.predict_proba(X_test_trans)[:, 1]

print("f1:", f1_score(y_test, y_pred))
print("precision:", precision_score(y_test, y_pred))
print("recall:", recall_score(y_test, y_pred))
print("roc_auc:", roc_auc_score(y_test, y_proba))

In [ ]:
feature_names = preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": cat_model.feature_importances_
})

importance_df.sort_values("importance", ascending=False).head(30)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "model__depth": [4, 5, 6, 7],
    "model__learning_rate": [0.03, 0.05, 0.07],
    "model__l2_leaf_reg": [3, 5, 7, 9],
    "model__iterations": [300, 500],
}

search = RandomizedSearchCV(
    estimator=cat_pipe,
    param_distributions=param_dist,
    n_iter=10,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    verbose=2,
    random_state=42
)

search.fit(X_sub, y_sub)

In [ ]:
print("Best Score:", search.best_score_)
print("Best Params:", search.best_params_)

In [ ]:
pd.DataFrame(search.cv_results_)[
    [
        'mean_test_score',
        'std_test_score',
        'param_model__depth',
        'param_model__learning_rate',
        'param_model__l2_leaf_reg',
        'param_model__iterations'
    ]
].sort_values(
    by='mean_test_score',
    ascending=False
).head(10)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import numpy as np

X_train, X_valid, y_train, y_valid = train_test_split(
    X_sub, y_sub,
    test_size=0.2,
    stratify=y_sub,
    random_state=42
)

best_model = search.best_estimator_
best_model.fit(X_train, y_train)

proba = best_model.predict_proba(X_valid)[:, 1]

best_f1 = 0
best_t = 0

for t in np.arange(0.20, 0.80, 0.01):
    pred = (proba >= t).astype(int)
    score = f1_score(y_valid, pred)

    if score > best_f1:
        best_f1 = score
        best_t = t

print("Best threshold:", best_t)
print("Best F1:", best_f1)

In [ ]:
shap.dependence_plot(
    "num__배아_이식_집중도",
    shap_values,
    X_test_trans,
    feature_names=feature_names,
    interaction_index=None
)